# Exercise 7: From the log to the lake

Every run of your job does three things: **read** the new records of the topic `logs`,
**write** them into the lake (Parquet files in `data/lake/`), **remember** where it stopped.

This notebook gives you the building blocks (`tools/lakelib.py`). You write one function,
`run_once()`. Save your final version of this notebook into your `solution/` folder.

In [1]:
import sys; sys.path.append("/home/jovyan/tools")
from lakelib import *

## 1. The example: a job that copies everything

It reads the topic from the beginning, every time. Run the next two cells and look at the
checker's last line.

In [2]:
def run_once():
    new = read_new(None)          # None = from the beginning of the topic
    write_files(new, LAKE)        # straight into the lake

reset()                           # empty lake
run_once()
check()

reset: lake, staging and state deleted
read_new: 3,800,000 new records
write_files: 3,800,000 records -> /home/jovyan/data/lake/logs/run-20261124-172217-964dc2


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Lake check — topic 'logs' (4 partitions)   2026-11-24 17:24:34
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 878,003)     878,003           0          0           0
        1           [0, 978,868)     978,868           0          0           0
        2           [0, 948,845)     948,845           0          0           0
        3           [0, 994,284)     994,284           0          0           0
-------------------------------------------------------------------------------
    total      3,800,000 records   3,800,000           0          0           0

files under /home/jovyan/data/lake: 4 data files, 142.7 MB, 35.68 MB per file on average
check took 3.0 s

Exactly once: PASS


{'topic': 3800000,
 'lake': 3800000,
 'dup': 0,
 'miss': 0,
 'lag': 0,
 'files': 4,
 'mb': 142.701491,
 'ok': True,
 'problems': []}

In [3]:
run_once()                        # the next run
check()

read_new: 3,800,000 new records
write_files: 3,800,000 records -> /home/jovyan/data/lake/logs/run-20261124-172434-849129


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Lake check — topic 'logs' (4 partitions)   2026-11-24 17:25:48
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 878,003)   1,756,006     878,003          0           0
        1           [0, 978,868)   1,957,736     978,868          0           0
        2           [0, 948,845)   1,897,690     948,845          0           0
        3           [0, 994,284)   1,988,568     994,284          0           0
-------------------------------------------------------------------------------
    total      3,800,000 records   7,600,000   3,800,000          0           0

files under /home/jovyan/data/lake: 8 data files, 285.4 MB, 35.68 MB per file on average
check took 3.1 s

Exactly once: FAIL — 3,800,000 duplicates


{'topic': 3800000,
 'lake': 7600000,
 'dup': 3800000,
 'miss': 0,
 'lag': 0,
 'files': 8,
 'mb': 285.402982,
 'ok': False,
 'problems': ['3,800,000 duplicates']}

## 2. The building blocks

| Block | What it does |
|---|---|
| `load_state()` | where the last run stopped, from a small state file (`None` if there is none) |
| `save_state(new.end)` | stores where this run stopped in the state file |
| `lake_position()` | where the last run stopped, read from the lake itself (`None` if the lake is empty) |
| `read_new(start)` | the records after `start`, up to the end of the topic *now* (`new.end`) |
| `write_files(new, folder)` | writes them as Parquet into a new run folder inside `LAKE` or `STAGING` |
| `publish(run)` | moves a finished run folder from `STAGING` into the lake: one rename, a single step |

`LAKE` is what readers see; `STAGING` is a folder next to it that nobody reads.
`new.df.show()` shows the records of a run as a Spark DataFrame.

## 3. Your architecture

Write `run_once()` for the architecture you chose (A, B or C, see the README).
The order of the calls *is* your architecture.

In [4]:
def run_once_a():
    start = load_state()  # stop point out state file
    new = read_new(start) # read niew record
    if new.count == 0:
        return
    write_files(new, LAKE) # write directly in the lake
    save_state(new.end)    # remebering it

In [5]:
def run_once_b():
    start = lake_position()  # stop point out state file
    new = read_new(start) # read niew record
    if new.count == 0:
        return
    run = write_files(new, STAGING) # write to staging 
    publish(run)    # in one rename to the lake

In [6]:
def run_once_c():
    start = load_state()
    new = read_new(start)
    if new.count == 0:
        return
    write_files(new, LAKE)
    save_state(new.end)

In [7]:
start = 0
new = read_new(start)
print(type(new))
print(new)
print([a for a in dir(new) if not a.startswith("_")])

import lakelib
print([f for f in dir(lakelib) if not f.startswith("_")])
help(read_new)

read_new: 3,800,000 new records
<class 'lakelib.NewRecords'>
NewRecords(3,800,000 records; start={0: 0, 1: 0, 2: 0, 3: 0}, end={0: 878003, 1: 978868, 2: 948845, 3: 994284})
['count', 'df', 'end', 'partition_df', 'start']
['BOOTSTRAP', 'DATA', 'DEDUP_SQL', 'LAKE', 'LAKE_SQL', 'NewRecords', 'SCHEMA', 'STAGING', 'STATE', 'SimulatedStop', 'TOPIC', 'add_records', 'check', 'check_all', 'get_spark', 'importlib', 'json', 'lake_position', 'load_state', 'os', 'publish', 'random', 'read_new', 'reset', 'save_state', 'shutil', 'subprocess', 'sys', 'time', 'uuid', 'write_files']
Help on function read_new in module lakelib:

read_new(start)
    The records after `start` ({partition: offset}, None = from the beginning), up to the end of the
    topic NOW. The end is fixed here: records that arrive later are left for the next run.



## 4. Check it

`check_all` empties the lake, then runs your job in five situations: a first run, new records,
nothing new, a run that stops **while** writing, a run that stops **after** writing.
It takes 2-4 minutes.

Architecture C: readers use a view, so check through it: `check_all(run_once, lake_sql=DEDUP_SQL)`.

In [8]:
check_all(run_once_a)

reset: lake, staging and state deleted


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             3,800,000           0        0           0  71.5s  PASS
2 new records arrive, run again         3,900,000           0        0           0   7.7s  PASS
3 nothing new, run again                3,900,000           0        0           0   0.1s  PASS
4 run stops WHILE writing: right after  3,924,944           0        0      75,056         PASS
  ... then the next run                 4,024,944      24,944        0           0   3.3s  FAIL
5 run stops AFTER writing: right after  4,124,944      24,944        0           0         FAIL
  ... then the next run                 4,224,944     124,944        0           0   7.3s  FAIL
-----------------------------------------------------------------------------------------------
Exactly once after every run: NO — see

[('1 first run',
  {'topic': 3800000,
   'lake': 3800000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 142.701491,
   'ok': True,
   'problems': []},
  71.49700093269348),
 ('2 new records arrive, run again',
  {'topic': 3900000,
   'lake': 3900000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 143.859719,
   'ok': True,
   'problems': []},
  7.651116847991943),
 ('3 nothing new, run again',
  {'topic': 3900000,
   'lake': 3900000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 143.859719,
   'ok': True,
   'problems': []},
  0.06699299812316895),
 ('4 run stops WHILE writing: right after',
  {'topic': 4000000,
   'lake': 3924944,
   'dup': 0,
   'miss': 0,
   'lag': 75056,
   'files': 9,
   'mb': 144.14951,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 4000000,
   'lake': 4024944,
   'dup': 24944,
   'miss': 0,
   'lag': 0,
   'files': 13,
   'mb': 145.308575,
   'ok': False,
   'problems': ['

In [9]:
check_all(run_once_b)

reset: lake, staging and state deleted

situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             4,100,000           0        0           0  65.5s  PASS
2 new records arrive, run again         4,200,000           0        0           0   7.8s  PASS
3 nothing new, run again                4,200,000           0        0           0   0.1s  PASS
4 run stops WHILE writing: right after  4,200,000           0        0     100,000         PASS
  ... then the next run                 4,300,000           0        0           0   5.0s  PASS
5 run stops AFTER writing: right after  4,300,000           0        0     100,000         PASS
  ... then the next run                 4,400,000           0        0           0   6.1s  PASS
-----------------------------------------------------------------------------------------------

[('1 first run',
  {'topic': 4100000,
   'lake': 4100000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 147.870678,
   'ok': True,
   'problems': []},
  65.47633528709412),
 ('2 new records arrive, run again',
  {'topic': 4200000,
   'lake': 4200000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 149.02606,
   'ok': True,
   'problems': []},
  7.784681797027588),
 ('3 nothing new, run again',
  {'topic': 4200000,
   'lake': 4200000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 149.02606,
   'ok': True,
   'problems': []},
  0.13896894454956055),
 ('4 run stops WHILE writing: right after',
  {'topic': 4300000,
   'lake': 4200000,
   'dup': 0,
   'miss': 0,
   'lag': 100000,
   'files': 8,
   'mb': 149.02606,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 4300000,
   'lake': 4300000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 12,
   'mb': 150.182708,
   'ok': True,
   'problems': []},
  5

In [10]:
check_all(run_once_c, lake_sql=DEDUP_SQL)

reset: lake, staging and state deleted


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             4,400,000           0        0           0  70.5s  PASS
2 new records arrive, run again         4,500,000           0        0           0   6.4s  PASS
3 nothing new, run again                4,500,000           0        0           0   0.0s  PASS
4 run stops WHILE writing: right after  4,552,935           0        0      47,065         PASS
  ... then the next run                 4,600,000           0        0           0   4.1s  PASS
5 run stops AFTER writing: right after  4,700,000           0        0           0         PASS
  ... then the next run                 4,700,000           0        0           0   5.4s  PASS
-----------------------------------------------------------------------------------------------
Exactly once after every run: YES


[('1 first run',
  {'topic': 4400000,
   'lake': 4400000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 153.03797,
   'ok': True,
   'problems': []},
  70.47537016868591),
 ('2 new records arrive, run again',
  {'topic': 4500000,
   'lake': 4500000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 154.192026,
   'ok': True,
   'problems': []},
  6.415771961212158),
 ('3 nothing new, run again',
  {'topic': 4500000,
   'lake': 4500000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 154.192026,
   'ok': True,
   'problems': []},
  0.013191699981689453),
 ('4 run stops WHILE writing: right after',
  {'topic': 4600000,
   'lake': 4552935,
   'dup': 0,
   'miss': 0,
   'lag': 47065,
   'files': 9,
   'mb': 154.742786,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 4600000,
   'lake': 4600000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 13,
   'mb': 155.89506,
   'ok': True,
   'problems': []},
  

In [11]:
help(lake_position); 

Help on function lake_position in module lakelib:

lake_position()
    Where the last run stopped, read from the lake: {partition: highest kafka_offset + 1}, or None.

